# Residue conservation preprocessing

Input data for Figures 3B and 4B


## Notebook overview
1. Downloads `ConSuf10k_PDBid_seq_cons.fasta` and `consurf10k_train_ids.txt` from Zenodo if not already present
2. Parses amino acid sequences and color conservation scores (1–9) for each protein
3. Filters to the training split only
4. Excludes proteins with length > 1022 residues (34 proteins, 41,769 residues) — these exceed the PLM input limit and are not used in the downstream analysis
5. Saves `conservation_seq.fasta` (one entry per protein, for PLM embedding)
6. Saves `conservation_features.csv` (per-residue features: conservation class)

In [ ]:
import os
from pathlib import Path

import urllib.request
import pandas as pd
import plotly.express as px

In [ ]:
_cwd = Path.cwd()
REPO_ROOT = next((str(p) for p in [_cwd, *_cwd.parents] if (p / ".git").exists()), str(_cwd))

ZENODO_FASTA  = os.path.join(REPO_ROOT, "examples/paper/data/conservation/raw/ConSuf10k_PDBid_seq_cons.fasta")
TRAIN_IDS     = os.path.join(REPO_ROOT, "examples/paper/data/conservation/raw/consurf10k_train_ids.txt")
OUTPUT_FASTA  = os.path.join(REPO_ROOT, "examples/paper/data/conservation/processed/conservation_seq.fasta")
OUTPUT_CSV    = os.path.join(REPO_ROOT, "examples/paper/data/conservation/processed/conservation_features.csv")
MAX_LEN       = 1022

## 1. Download Zenodo data

In [ ]:
ZENODO_BASE = "https://zenodo.org/records/5238537/files"

def _download(url, dest):
    os.makedirs(os.path.dirname(dest), exist_ok=True)
    if not os.path.exists(dest):
        print(f"Downloading {url} ...")
        urllib.request.urlretrieve(url, dest)
        print(f"saved to {dest}")
    else:
        print(f"Already present: {dest}")

_download(f"{ZENODO_BASE}/ConSuf10k_PDBid_seq_cons.fasta", ZENODO_FASTA)
_download(f"{ZENODO_BASE}/consurf10k_train_ids.txt",         TRAIN_IDS)

## 2. Parse sequences and conservation scores

The Zenodo FASTA has four entries per protein (sequence, log conservation, color conservation, blank). This cell extracts the amino acid sequence and the integer color conservation scores (1–9) for each protein, normalising the chain identifier to `{pdb_id}-{chain}` format.

In [ ]:
proteins = {}  # chain_id -> {"sequence": str, "conservation": list[int]}

with open(ZENODO_FASTA, "r") as fh:
    cur_id   = None
    cur_mode = None  # "seq" | "color"
    for line in fh:
        line = line.rstrip()
        if not line:
            continue
        if line.startswith(">"):
            header = line[2:].strip()   # strip leading "> "
            parts  = header.split()
            pdb_id = parts[0]
            chain  = parts[1] if len(parts) > 1 else ""
            chain_id = f"{pdb_id}-{chain}"
            suffix = " ".join(parts[2:]) if len(parts) > 2 else ""
            if "color conservation" in suffix:
                cur_id   = chain_id
                cur_mode = "color"
            elif "log conservation" in suffix:
                cur_mode = None  # skip log scores
            else:
                cur_id   = chain_id
                cur_mode = "seq"
                proteins.setdefault(chain_id, {"sequence": "", "conservation": []})
        else:
            if cur_mode == "seq" and cur_id:
                proteins[cur_id]["sequence"] += line
            elif cur_mode == "color" and cur_id:
                proteins[cur_id]["conservation"] = [int(x) for x in line.split(",")]
                cur_mode = None

print(f"Total proteins parsed: {len(proteins)}")
sample_id = next(iter(proteins))
sample    = proteins[sample_id]
print(f"Example — {sample_id}: {len(sample['sequence'])} aa, "
      f"{len(sample['conservation'])} conservation scores")

## 3. Filter to training split

Load `consurf10k_train_ids.txt` and keep only those proteins. The IDs are in `{pdb_id}-{chain}` format, one per line.

In [ ]:
with open(TRAIN_IDS) as fh:
    train_ids = {line.strip() for line in fh if line.strip()}

proteins_train = {pid: data for pid, data in proteins.items() if pid in train_ids}
print(f"Training proteins: {len(proteins_train)} / {len(proteins)} total")
print(f"IDs in train_ids not found in FASTA: "
      f"{len(train_ids - set(proteins_train))}")

## 4. Filter to proteins ≤ MAX_LEN and save FASTA

Proteins with length > 1022 residues are excluded: they exceed the context window of the
PLMs used for embedding and are not included in the downstream analysis.
9,358 of the 9,392 training proteins pass this filter (34 excluded, 41,769 residues dropped).

In [ ]:
proteins_short = {
    pid: data
    for pid, data in sorted(proteins_train.items())
    if len(data["sequence"]) <= MAX_LEN
}

n_excluded = len(proteins_train) - len(proteins_short)
print(f"Proteins ≤ {MAX_LEN} aa (kept):    {len(proteins_short)}")
print(f"Proteins > {MAX_LEN} aa (excluded): {n_excluded}")

os.makedirs(os.path.dirname(OUTPUT_FASTA), exist_ok=True)
with open(OUTPUT_FASTA, "w") as fh:
    for pid, data in proteins_short.items():
        fh.write(f">{pid}\n{data['sequence']}\n")
print(f"Saved {len(proteins_short)} sequences to {OUTPUT_FASTA}")

## 5. Class distribution

In [ ]:
rows = []
for pid, data in proteins_short.items():
    pdb_id = pid.split("-")[0]
    for pos, score in enumerate(data["conservation"]):
        residue_id = pos + 1
        rows.append({
            "residue_name":     f"{pid}-{residue_id}",
            "protein_chain_id": pid,
            "protein_id":       pdb_id,
            "residue_id":       residue_id,
            "conservation":     f"{score}-conservation",
        })

df_residues = pd.DataFrame(rows)
os.makedirs(os.path.dirname(OUTPUT_CSV), exist_ok=True)
df_residues.to_csv(OUTPUT_CSV, index=False)
print(f"Saved {len(df_residues):,} residue rows to {OUTPUT_CSV}")
df_residues.head()